In [1]:
# Install deps
!pip uninstall -y sympy transformers sentence-transformers
!pip install sympy==1.12 transformers==4.38.2 sentence-transformers==2.6.1
!pip install pinecone nltk tiktoken -q

# Env config
%env PINECONE_API_KEY=pcsk_5zmtAT_Hf7BPztEgkqyeNs9Lb9kHoihUxmi3ZKT1CN14gSLwPxryDiPmYGDrT9eoyEkR8z
%env INDEX_NAME=knowledge-base-1

Found existing installation: sympy 1.14.0
Uninstalling sympy-1.14.0:
  Successfully uninstalled sympy-1.14.0
Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0
Found existing installation: sentence-transformers 5.2.2
Uninstalling sentence-transformers-5.2.2:
  Successfully uninstalled sentence-transformers-5.2.2
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.7/130.7 kB 5.9 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of torch to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of torch to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 76.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 89.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.3/163.3 kB 12.3 MB/s eta 0:00:00
   ━

In [2]:
# Initalise tokenizers


from pinecone import Pinecone, ServerlessSpec
import os, json, re
from typing import Dict, List, Any, Iterable, Tuple

import nltk
from nltk.tokenize import sent_tokenize
import tiktoken

from sentence_transformers import SentenceTransformer

nltk.download("punkt")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [3]:
# Load the glossary
from google.colab import drive
drive.mount('/content/drive')

GLOSSARY_PATH = "/content/drive/MyDrive/Polaris/Knowledge Base 2.0/abbrev_glossary.json"

with open(GLOSSARY_PATH, "r", encoding="utf-8") as f:
    GLOSSARY: Dict[str, str] = json.load(f)

print(f"Loaded glossary entries: {len(GLOSSARY)}")

Mounted at /content/drive
Loaded glossary entries: 98


In [5]:
# Initialise Pinecone connection (API key and Index name)
import os

pc = Pinecone(api_key=f"{os.getenv("PINECONE_API_KEY")}")

INDEX_NAME = os.getenv("INDEX_NAME")
DIMENSIONS = 768

if not pc.has_index(INDEX_NAME):
    pc.create_index(
        name=INDEX_NAME,
        dimension=DIMENSIONS,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )

index = pc.Index(INDEX_NAME)
print(index.describe_index_stats())

{'_response_info': {'raw_headers': {'connection': 'keep-alive',
                                    'content-length': '150',
                                    'content-type': 'application/json',
                                    'date': 'Wed, 11 Feb 2026 10:17:39 GMT',
                                    'grpc-status': '0',
                                    'server': 'envoy',
                                    'x-envoy-upstream-service-time': '45',
                                    'x-pinecone-request-latency-ms': '43',
                                    'x-pinecone-response-duration-ms': '46'}},
 'dimension': 768,
 'index_fullness': 0.0,
 'memoryFullness': 0.0,
 'metric': 'cosine',
 'namespaces': {},
 'storageFullness': 0.0,
 'total_vector_count': 0,
 'vector_type': 'dense'}


In [ ]:
# Load embedding model and create text helpers

model = SentenceTransformer("all-mpnet-base-v2")


# Uses glossary to expand abbrevated words for better retrieval [e.g. EP becomes EP (Exchange Participant)]
def expand_abbreviations(text: str, glossary: Dict[str, str]) -> str:
    if not text:
        return text

    keys = sorted(glossary.keys(), key=len, reverse=True)
    for k in keys:
        meaning = glossary[k]
        text = re.sub(rf"\b{re.escape(k)}\b", f"{k} ({meaning})", text)
    return text


# Normalise the role requirements, returns none if we need to skip it
def normalize_req_value(v):
    if v is None:
        return None
    s = str(v).strip()
    if not s:
        return None

    if s.lower() in {"non-existent", "non existent", "n/a", "na", "none", "null", "-"}:
        return None

    return s


# Build a doc string for a competency (the 'text' section of the embedding)
def competency_to_doc_text(c: Dict[str, Any], glossary: Dict[str, str]) -> str:
    lines = []
    lines.append(f"Function: {c.get('function_name','')} | Code: {c.get('function_code','')}")
    lines.append(f"Skill Category: {c.get('skill_category','')}")
    lines.append(f"Skill Type: {c.get('skill_type','')}")
    lines.append(f"Competency: {c.get('competency_name','')} [{c.get('competency_id','')}]")
    lines.append(f"Description: {c.get('description','')}")

    cv = c.get("cv_keywords") or []
    uc = c.get("aiesec_use_cases") or []
    if isinstance(cv, list) and cv:
        lines.append(f"CV Keywords: {', '.join(cv)}")
    if isinstance(uc, list) and uc:
        lines.append(f"AIESEC Use-Cases: {', '.join(uc)}")

    slb = c.get("skill_level_behaviors") or {}
    if isinstance(slb, dict) and slb:
        lines.append("Skill Level Behaviors:")
        for lvl in ["Beginner", "Intermediate", "Advanced", "Expert"]:
            if lvl in slb and slb[lvl]:
                lines.append(f"- {lvl}: {slb[lvl]}")

    rlr = c.get("role_level_requirements") or {}
    if isinstance(rlr, dict) and rlr:
        lines.append("Role Level Requirements:")
        for role, req in rlr.items():
            clean = normalize_req_value(req)
            if clean is None:
                continue
            lines.append(f"- {role}: {clean}")

    src = c.get("source") or {}
    if isinstance(src, dict) and src:
        lines.append(f"Source: {src.get('workbook','')} / {src.get('sheet','')}")

    raw_text = "\n".join(lines).strip()
    expanded = expand_abbreviations(raw_text, glossary)

    return raw_text + "\n\n--- Expanded Abbreviations ---\n\n" + expanded


# Tokenizes the entire text, then creates token windows, this allows for better ingestion
def chunk_by_tokens(
    text: str,
    max_tokens: int = 350,
    overlap_tokens: int = 60,
    encoding_name: str = "cl100k_base",
) -> List[str]:
    enc = tiktoken.get_encoding(encoding_name)
    tokens = enc.encode(text)

    if len(tokens) <= max_tokens:
        return [text]

    chunks = []
    start = 0
    while start < len(tokens):
        end = min(start + max_tokens, len(tokens))
        window = tokens[start:end]
        chunks.append(enc.decode(window))

        if end == len(tokens):
            break
        start = max(0, end - overlap_tokens)

    return chunks


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
# Load the competencies and create metadata

def load_competencies_from_file(path: str) -> List[Dict[str, Any]]:

    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    if isinstance(data, list):
        return data

    raise ValueError(f"Unrecognized JSON format in {path}")


def competency_to_metadata(c: Dict[str, Any], file_name: str) -> Dict[str, Any]:
    src = c.get("source") or {}
    return {
        "file_name": file_name,
        "function_code": c.get("function_code", ""),
        "function_name": c.get("function_name", ""),
        "skill_category": c.get("skill_category", ""),
        "skill_type": c.get("skill_type", ""),
        "competency_id": c.get("competency_id", ""),
        "competency_name": c.get("competency_name", ""),
        "source_workbook": (src.get("workbook") if isinstance(src, dict) else ""),
        "source_sheet": (src.get("sheet") if isinstance(src, dict) else ""),
    }

In [ ]:
# Ingest competencies and then do a stable upsert

COMPETENCY_FOLDER = "/content/drive/MyDrive/Polaris/Knowledge Base 2.0/JSONs"
json_files = [f for f in os.listdir(COMPETENCY_FOLDER) if f.endswith(".json")]
print(f"Found {len(json_files)} files: {json_files}")

def batched(it: List[Any], batch_size: int) -> Iterable[List[Any]]:
    for i in range(0, len(it), batch_size):
        yield it[i:i+batch_size]

vectors_to_upsert = []
total_chunks = 0

for file_name in json_files:
    path = os.path.join(COMPETENCY_FOLDER, file_name)
    competencies = load_competencies_from_file(path)

    for c in competencies:
        doc = competency_to_doc_text(c, GLOSSARY)

        chunks = chunk_by_tokens(doc, max_tokens=350, overlap_tokens=60)

        base_meta = competency_to_metadata(c, file_name)

        for j, chunk in enumerate(chunks):
            vec = model.encode(chunk, convert_to_numpy=True).tolist()

            comp_id = c.get("competency_id") or c.get("competency_name") or "unknown"
            vid = f"{file_name}:{comp_id}:{j}"

            vectors_to_upsert.append({
                "id": vid,
                "values": vec,
                "metadata": {
                    **base_meta,
                    "chunk_index": j,
                    "text": chunk
                }
            })
            total_chunks += 1

    print(f"✅ Prepared {file_name}: {len(competencies)} competencies → {total_chunks} total chunks so far")

print(f"\nUpserting {len(vectors_to_upsert)} vectors into '{INDEX_NAME}'...")
for batch in batched(vectors_to_upsert, batch_size=200):
    index.upsert(vectors=batch)

print("✅ Done.")
print(index.describe_index_stats())

Found 7 files: ['ops_competencies.json', 'im_competencies.json', 'tm_competencies.json', 'mgmt_competencies.json', 'fin_competencies.json', 'mkt_competencies.json', 'sales_competencies.json']
✅ Prepared ops_competencies.json: 22 competencies → 44 total chunks so far
✅ Prepared im_competencies.json: 23 competencies → 90 total chunks so far
✅ Prepared tm_competencies.json: 20 competencies → 130 total chunks so far
✅ Prepared mgmt_competencies.json: 21 competencies → 172 total chunks so far
✅ Prepared fin_competencies.json: 19 competencies → 210 total chunks so far
✅ Prepared mkt_competencies.json: 28 competencies → 265 total chunks so far
✅ Prepared sales_competencies.json: 25 competencies → 315 total chunks so far

Upserting 315 vectors into 'knowledge-base-1'...
✅ Done.
{'_response_info': {'raw_headers': {'connection': 'keep-alive',
                                    'content-length': '185',
                                    'content-type': 'application/json',
                      

In [ ]:
# Retrival test

query_text = "Identify financial risks and mitigate losses"
qvec = model.encode(query_text, convert_to_numpy=True).tolist()

res = index.query(vector=qvec, top_k=5, include_metadata=True)

for m in res.get("matches", []):
    md = m.get("metadata", {})
    print("\n--- Match ---")
    print("Score:", m.get("score"))
    print("Competency:", md.get("competency_name"), "|", md.get("competency_id"))
    print("Function:", md.get("function_name"), "(", md.get("function_code"), ")")
    print("Skill Type:", md.get("skill_type"))
    print("Source:", md.get("source_sheet"))
    print("Chunk index:", md.get("chunk_index"))
    print("Text preview:\n", (md.get("text","")[:400] + "...") if md.get("text") else "(no text stored)")


--- Match ---
Score: 0.5952425
Competency: Risk Management | fin_hard_risk_management
Function: Finance ( FIN )
Skill Type: Hard
Source: FIN Functional Skills
Chunk index: 0
Text preview:
 Function: Finance | Code: FIN
Skill Category: Functional
Skill Type: Hard
Competency: Risk Management [fin_hard_risk_management]
Description: Identifying, assessing, and mitigating financial and operational risks
CV Keywords: Risk Control, Contingency Planning
AIESEC Use-Cases: Anticipating payment delays, planning emergency budget, potential financial losses
Skill Level Behaviors:
- Beginner: Esc...

--- Match ---
Score: 0.535036147
Competency: Problem-Solving/Analytical thinking | fin_soft_problem-solving_analytical_thinking
Function: Finance ( FIN )
Skill Type: Soft
Source: FIN Functional Skills
Chunk index: 0
Text preview:
 Function: Finance | Code: FIN
Skill Category: Functional
Skill Type: Soft
Competency: Problem-Solving/Analytical thinking [fin_soft_problem-solving_analytical_thinking]
Descr

In [ ]:
# Query retreival test

query_text = "What soft skills do I need to move from a TL to LCVP in finance?"
qvec = model.encode(query_text, convert_to_numpy=True).tolist()

res = index.query(vector=qvec, top_k=10, include_metadata=True)

for m in res["matches"]:
    md = m["metadata"]
    print("\nScore:", m["score"])
    print("Competency:", md.get("competency_name"), "|", md.get("competency_id"))
    print("Function:", md.get("function_name"), md.get("function_code"))
    print("Preview:", (md.get("text","")))



Score: 0.570217133
Competency: Cash Flow Management | fin_hard_cash_flow_management
Function: Finance FIN
Preview:  Ensures liquidity for day-to-day operations.
- Advanced: Optimizes cash flows for multiple projects/events.
- Expert: Builds reserves, sustainability funds, and investment strategies.
Role Level Requirements:
- Level 2 : TL (Team Leader): Beginner
- Level 3 : LCVP (Local Committee Vice-President) / EST (Entity Support Team): Intermediate
- Level 4 : MCVP (Member Committee Vice-President) / GST (Global Support Team): Advanced
- Level 4 : AI (AIESEC International): Expert
Source: Competency Models.xlsx / FIN (Finance) Functional Skills

Score: 0.565711081
Competency: Adaptability | fin_soft_adaptability
Function: Finance FIN
Preview: Function: Finance | Code: FIN
Skill Category: Functional
Skill Type: Soft
Competency: Adaptability [fin_soft_adaptability]
Description: Navigating new systems, processes, or regulation changes
CV Keywords: Agility, Fast Learner, Resilience
AIE